# Notebook 1: Data Loading and Cleaning

## Objective
Load and clean the EBA 2025 Transparency Excercise dataset to extract some key banking indicators for the Early Warning System analysis.

## Data source
EBA EU-wide Transparency Exercise 2025, publicly available at eba.europa.eu
Coverage: 120 European banks across 25 countries, 4 different observations in time (September 2024, December 2024, March 2025, June 2025)

In [8]:
import pandas as pd
import numpy as np

## 1. Data Loading and Initial Exploration

We start by understanding the structure of the raw dataset.
It is in long format: each row represents one indicator for one bank in one period, rather than one row per bank.

In [ ]:
# Load main data file (capital, leverage, P&L, assets)
oth = pd.read_csv("tr_oth.csv", sep=",", quotechar='"', low_memory=False)
meta = pd.read_excel("TR_Metadata.xlsx", sheet_name="List of Institutions", header=1)

In [ ]:
print(oth.shape) # rows x columns
print(sorted(oth['Period'].unique()))
print(oth['LEI_Code'].nunique())
print(meta.columns[:8].tolist())
print(len(meta))

(91277, 15)
[np.int64(202409), np.int64(202412), np.int64(202503), np.int64(202506)]
120
['Country', 'Desc_country', 'LEI_Code', 'Name', 'Finrep', 'Fin_year_end', 'Footnotes', 'TE_24']
120


## 2. Extraction of Key Indicators
We filter the dataset to retain only relevant indicators: CET1 capital, Risk-Weighted Assets, leverage ratio and total assets.

In [14]:
# Extract CET1 Capital
cet1 = oth[oth['Label'] == 'COMMON EQUITY TIER 1 CAPITAL (net of deductions and after applying transitional adjustments)'][['LEI_Code', 'Period', 'Amount']].rename(columns = {'Amount':'CET1_capital'})

# Extract Total Risk Exposure Amount (denominator for CET1 ratio)
rwa = oth[oth['Label'] == 'TOTAL RISK EXPOSURE AMOUNT'][['LEI_Code', 'Period', 'Amount']].rename(columns={'Amount':'RWA'})

# Extract Leverage Ratio
lev = oth[oth['Label'] == 'Leverage ratio - using a fully phased-in definition of Tier 1 capital'][['LEI_Code', 'Period', 'Amount']].rename(columns={'Amount': 'leverage_ratio'})

# Extract Total Assets
assets = oth[oth['Label'] == 'Total Assets'][['LEI_Code', 'Period', 'Amount']].rename(columns={'Amount':'total_assets'})

print(len(cet1))
print(len(rwa))
print(len(lev))
print(len(assets))

469
469
469
426


In [16]:
# Merge all indicators into a single dataframe
df = cet1.merge(rwa, on=['LEI_Code', 'Period'], how='outer')
df = df.merge(lev, on=['LEI_Code', 'Period'], how='outer')
df = df.merge(assets, on=['LEI_Code', 'Period'], how='outer')

# Add bank name and country from metadata
df = df.merge(meta[['LEI_Code', 'Name', 'Country']], on='LEI_Code', how='left')

# Convert Amount columns to numeric
df['CET1_capital']=pd.to_numeric(df['CET1_capital'], errors='coerce')
df['RWA']=pd.to_numeric(df['RWA'], errors='coerce')
df['leverage_ratio']=pd.to_numeric(df['leverage_ratio'], errors='coerce')
df['total_assets']=pd.to_numeric(df['total_assets'], errors='coerce')

# Calculate CET1 ratio (%)
df['CET1_ratio'] =(df['CET1_capital']/df['RWA']) * 100

# Convert Period to datetime
df['Period'] = pd.to_datetime(df['Period'].astype(str), format='%Y%m')

print(df.shape)
print(df.head())
print(df.isnull().sum())

(474, 9)
               LEI_Code     Period  CET1_capital           RWA  \
0  0W2PZJM8XOY22M4GG883 2024-09-01   5667.386936  30769.686172   
1  0W2PZJM8XOY22M4GG883 2024-12-01   6103.541436  30814.453610   
2  0W2PZJM8XOY22M4GG883 2025-03-01   6193.371301  28815.090686   
3  0W2PZJM8XOY22M4GG883 2025-06-01   6102.372815  28965.357494   
4  2138008AVF4W7FMW8W87 2024-09-01    954.745084   4975.646909   

   leverage_ratio  total_assets                            Name Country  \
0        0.075914  94984.876492  DekaBank Deutsche Girozentrale      DE   
1        0.081894  92935.124241  DekaBank Deutsche Girozentrale      DE   
2             NaN  94158.314784  DekaBank Deutsche Girozentrale      DE   
3             NaN  96079.978435  DekaBank Deutsche Girozentrale      DE   
4        0.048261  18596.111717                     CEC BANK SA      RO   

   CET1_ratio  
0   18.418735  
1   19.807398  
2   21.493499  
3   21.067832  
4   19.188361  
LEI_Code            0
Period              0
CET

## 3. Data Cleaning
We remove rows with missing bank names and missing CET1 ratio values.
The leverage ratio column is dropped due to the excessive missing values (more than 50%).

In [17]:
# Drop rows with missing bank name (unmatched LEI codes)
df = df.dropna(subset=['Name', 'Country'])

# Drop rows with missing CET1 ratio (core indicator)
df = df.dropna(subset=['CET1_ratio'])

# Drop leverage ratio column (too many missing values)
df = df.drop(columns=['leverage_ratio'])

# Reset index
df = df.reset_index(drop=True)

# Check
print(df.shape)
print(df.isnull().sum())
print(df['Country'].value_counts())

(465, 8)
LEI_Code         0
Period           0
CET1_capital     0
RWA              0
total_assets    48
Name             0
Country          0
CET1_ratio       0
dtype: int64
Country
DE    92
IT    48
FR    44
ES    40
SE    24
NL    22
BE    20
IE    20
AT    20
GR    16
PT    12
DK    12
FI    12
NO     9
RO     8
SI     8
MT     8
HU     8
LT     8
PL     8
LU     8
LI     6
LV     4
EE     4
CY     4
Name: count, dtype: int64


## 4. Output
The cleaned dataset included 465 observations across 119 banks and 25 countries. It is saved as banking_indicators_clean.csv for use in subsequent notebooks.

In [19]:
# Saving clean dataset
df.to_csv("banking_indicators_clean.csv", index=False)

print(df.shape)
print(df['Name'].nunique())
print(df['Period'].unique())
print(df['Country'].nunique())


(465, 8)
119
<DatetimeArray>
['2024-09-01 00:00:00', '2024-12-01 00:00:00', '2025-03-01 00:00:00',
 '2025-06-01 00:00:00']
Length: 4, dtype: datetime64[us]
25
